# gramlab — chat
Ask the model questions. **Interactive use only** (widgets don't run in Save & Run All). **Accelerator: None**.
Inputs (*Add Input → Your Work → Notebooks*): **gramlab** (required: 25M writer + stage-3 index); optional **gramlab-writer**
(fine-tuned 0.5B writer — fluent answers), **gramlab-hybrid** (BM25 + meaning-map retriever), **gramlab-fullwiki** (all of Wikipedia).
Then **Run All**, wait ~1 min, type in the box.

Main function: `ask(question)`. Options: `retriever='bm25'|'hybrid'`, `writer='hf'|'small'`, `k=2` for two passages.

In [ ]:
import os, sys, glob, subprocess, json, numpy as np, torch, ipywidgets as w
from IPython.display import display
if not os.path.exists('/kaggle/working/gramlab'):
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/furevych/gramlab.git', '/kaggle/working/gramlab'], check=True)
sys.path.insert(0, '/kaggle/working/gramlab')
cands = sorted(glob.glob('/kaggle/input/**/work/runs/small_wiki_rag/ckpt.pt', recursive=True),
               key=lambda p: -torch.load(p, map_location='cpu', weights_only=False)['step'])
if not cands:
    raise SystemExit('Add the gramlab notebook output as an input (right panel -> Input -> Add Input -> Your Work)')
os.chdir(cands[0].split('/work/')[0])
from gramlab.ckpt import load_checkpoint
from gramlab.tokenizer import USER, ASSISTANT, END, EOS
from gramlab.retrieval import BM25Index, format_rag_user
model, tok, state = load_checkpoint('work/runs/small_wiki_rag/ckpt.pt', 'cpu')
full = sorted(glob.glob('/kaggle/input/**/work/bm25_full/weights.npz', recursive=True))
index = BM25Index.load(full[0].split('/weights.npz')[0] if full else 'work/bm25')
print(f"25M writer step {state['step']}, BM25 over {len(index):,} passages ({'full Wikipedia' if full else 'stage-3 index'})")

# optional: the fine-tuned 0.5B writer (gramlab-writer output) — fluent answers, ~10 s per reply on CPU
HF = None
hfd = sorted(glob.glob('/kaggle/input/**/work/runs/writer_hf/gramlab.json', recursive=True))
if hfd:
    from transformers import AutoTokenizer, AutoModelForCausalLM
    d = os.path.dirname(hfd[0]); torch.set_num_threads(4)
    HF = (AutoTokenizer.from_pretrained(d), AutoModelForCausalLM.from_pretrained(d, torch_dtype=torch.float32).eval(), json.load(open(hfd[0]))['system'])
    print('0.5B writer loaded; default writer = hf')

# optional: the meaning map (gramlab-map output) -> BM25 + map hybrid retriever (rule from hybrid.json if tuned)
HYB = None
m2 = glob.glob('/kaggle/input/**/results/map2.json', recursive=True)
if m2:
    from gramlab.map import HybridRetriever
    mroot = m2[0].split('/results/')[0]
    best = json.load(open(m2[0]))['best_for_hybrid']
    mdir = f'{mroot}/work/map2/{best}'
    try:
        import spacy; NLP = spacy.load('en_core_web_sm', disable=['parser', 'ner', 'lemmatizer'])
    except Exception:
        NLP = None
    def tag(q):
        if NLP is None: return (q.split(), ['NOUN'] * len(q.split()))
        d = NLP(q); return ([t.text for t in d], [t.pos_ for t in d])
    HYB = HybridRetriever.load(index, mdir, tag)
    if HYB.P.shape[0] != len(index):
        print(f'meaning map skipped: its passage matrix ({HYB.P.shape[0]:,}) was built for a different index ({len(index):,} passages)'); HYB = None
    else:
        print(f'meaning map loaded ({best}, weight {HYB.weight}, guard {HYB.guard}); default retriever = hybrid')
DEFAULT = 'hybrid' if HYB is not None else 'bm25'

def retrieve(q, k=1, retriever=None):
    retriever = retriever or DEFAULT
    if retriever == 'hybrid' and HYB is not None:
        return HYB.retrieve(q, k=k, max_words=90)
    return index.retrieve(q, k=k, max_words=90)

def ask(q, k=1, retriever=None, writer=None, show_passage=True, temperature=0.3):
    p = retrieve(q, k, retriever) or [{'title': '(no match)', 'text': 'No passage found.', 'score': 0}]
    if show_passage:
        for x in p: print(f"  [read: {x['title']}]")
    writer = writer or ('hf' if HF is not None else 'small')
    if writer == 'hf' and HF is not None:
        htok, hmodel, system = HF
        chat = [{'role': 'system', 'content': system}, {'role': 'user', 'content': format_rag_user(p, q)}]
        ids = htok(htok.apply_chat_template(chat, tokenize=False, add_generation_prompt=True), return_tensors='pt', add_special_tokens=False).input_ids
        with torch.no_grad():
            out = hmodel.generate(ids, max_new_tokens=80, do_sample=False, repetition_penalty=1.1, pad_token_id=htok.pad_token_id)
        return htok.decode(out[0, ids.shape[1]:], skip_special_tokens=True).strip()
    ids = [USER] + tok.encode(format_rag_user(p, q)) + [END, ASSISTANT]
    out = model.generate(torch.tensor([ids[-(model.ctx - 1):]]), max_new_tokens=80, temperature=temperature,
                         top_k=40, top_p=0.9, repetition_penalty=1.2, stop_ids=(END, EOS))
    return tok.decode(out[0, len(ids):].tolist())

box = w.Text(placeholder='ask about a place, a person, a thing...', layout=w.Layout(width='70%'), continuous_update=False)
btn, out = w.Button(description='Send'), w.Output()
def on_send(_=None):
    q = box.value.strip()
    if not q: return
    box.value = ''
    with out:
        print('you>  ', q); print('model>', ask(q)); print()
btn.on_click(on_send)
box.observe(lambda ch: on_send() if ch['name'] == 'value' and ch['new'] else None, names='value')
display(w.HBox([box, btn]), out)

In [ ]:
# the main function, without the widget — one question per cell
ask("Who was Winston Churchill?")